# The data

This is the first in a short series of notebooks introducing `matched`: how the input data is shaped, how it is cleaned, and how the allocation algorithm itself works. We start with the data supervisors and students provide.

### `projects.csv`

In [ ]:
import pathlib

import pandas as pd

raw = pathlib.Path("./raw")

projects = pd.read_csv(raw / "projects.csv").set_index("code")

projects.head(3)

`projects.csv` contains data collected from supervisors. Each project is given a unique code (`code (str)`). For each project code, we record:
- `external (bool)` indicating whether the project is external or internal
- `nmax (int>0)` - the maximum number of students that can be assigned to the project
- `course_i (bool)` - which course(s) the project is available for.

From the data above, we can see that the project `mabe-007` is an internal project that can be assigned to at most 3 students, and is available to `course1` only.

### `internal-choices.csv`

In [ ]:
choices_wide = pd.read_csv(raw / "internal-choices.csv").set_index("username")

choices_wide.head(3)

Each student fills in a form where they select $N$ projects they are interested in, in order of preference - one column per rank (`choice1`, `choice2`, ...). For each student, we also record which `course` they are enrolled on, and their `score` (e.g. academic performance), used as a tiebreaker when multiple students want the same project.

### Building `matched`'s input

`matched` doesn't take DataFrames - it works with plain Python `dict`s, so there's no implicit column-name/index contract to get right. It needs five dicts, each keyed by `username` or project `code`:

- `choices`: `{username: [code, ...]}` - a student's picks in preference order. Rank is just the list position, so we only need to drop the `NaN` cells below - no explicit rank number required.
- `scores`: `{username: score}`
- `courses`: `{username: course}`
- `nmax`: `{code: max_capacity}`
- `eligible_courses`: `{code: [course, ...]}` - which courses may pick each project

We build these directly from the two DataFrames above:

In [ ]:
choice_cols = [c for c in choices_wide.columns if c.startswith("choice")]

choices = {
    username: [code for code in row[choice_cols] if pd.notna(code)]
    for username, row in choices_wide.iterrows()
}
scores = choices_wide.score.to_dict()
courses = choices_wide.course.to_dict()

nmax = projects.nmax.to_dict()
eligible_courses = {
    code: [c for c in ("course1", "course2") if row[c]]
    for code, row in projects.iterrows()
}

(
    choices["tb241"],
    scores["tb241"],
    courses["tb241"],
    nmax["mabe-007"],
    eligible_courses["mabe-007"],
)

Each student now has an ordered list of project codes in `choices`, with matching entries in `scores`/`courses`, and each project has an entry in `nmax`/`eligible_courses`. These are the five shapes `matched.preprocess` and `matched.match` expect. In [Preprocessing](02-preprocessing.ipynb), we clean this data before allocating.